# PointBigBird-JEPA — visual walkthrough

A new sparse-input self-supervised architecture. Three replacements vs. the
v1–v8 OmniField backbone:

| Old (v1–v8 OmniField)                | New (PointBigBird)                       |
|--------------------------------------|------------------------------------------|
| Perceiver cascade with learnable latents | Per-token Transformer encoder       |
| `latent_pos` + Gaussian attention bias | Space-filling-curve serialization (z, z_rev, Hilbert, Hilbert_rev) |
| Dense cross-attention from a small latent set | **BigBird block-sparse** self-attention |
| Deterministic Gaussian soft-pool target | Direct lookup: target_coords ⊂ pool_coords |

Kept from v8 (the load-bearing parts):
- EMA target encoder (m: 0.999 → 1.0)
- DINO-style EMA centering on target features
- smooth-L1 distance
- **i-JEPA multi-block disjoint masking** — 4 contiguous target blocks + 1
  context block, all disjoint within the 40 % pool.

Outline:
1. Tokenization (RGB + γ-features)
2. The 4 serialization orders
3. Per-sample subset orderings + how a context chunk reorders under each curve
4. BigBird block-sparse attention pattern
5. Equivalence sanity: BigBird in `full` mode matches dense MHA exactly
6. Per-layer random shuffling in the encoder
7. End-to-end forward pass on one CIFAR-10 image
8. JEPA loss + diagnostics


## 0. Setup

In [ ]:
import os, sys, math, time, copy
# Make the local `pbb` package importable
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

import torchvision
from torchvision import transforms

from pbb import (
    PBBConfig,
    morton_code_2d, hilbert_code_2d,
    precompute_grid_orderings, subset_perm, invert_perm, precompute_subset_orderings,
    MultiHeadAttention, BigBirdSparseAttention,
    Tokenizer, EncoderBlock, PBBEncoder, PBBPredictor,
    PBBChunkCIFAR10, orderings_from_batch, CIFAR_CLASSES,
    TargetCenter, ema_update, gather_target_features, jepa_loss,
    short_params,
)

torch.manual_seed(0); np.random.seed(0)
cfg = PBBConfig()
print(f"K_POOL={cfg.k_pool}  K_CTX={cfg.k_ctx}  N_TGT={cfg.n_tgt}  "
      f"D_model={cfg.d_model}  block_size={cfg.block_size}")
print(f"BigBird: window={cfg.window}  random={cfg.n_random}  global={cfg.n_global}  "
      f"=> {2*cfg.window+1 + cfg.n_random + cfg.n_global} blocks attended per query block")


## 1. Tokenization

Each input point becomes a single D-dim token:
$$
\text{token}_n = W_\text{signal} \cdot \text{rgb}_n \;+\; W_\text{pos} \cdot \gamma(\text{coord}_n)
$$
where `γ(c) = [sin(B·c), cos(B·c)]` are Gaussian Fourier features.


In [ ]:
# Load one CIFAR-10 image
transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.5,)*3, (0.5,)*3)])
cifar = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
img, label = cifar[7]
print(f"label = {CIFAR_CLASSES[label]}, image shape = {tuple(img.shape)}")

# Build coords
ys, xs = torch.meshgrid(
    torch.linspace(-1.0, 1.0, cfg.image_size),
    torch.linspace(-1.0, 1.0, cfg.image_size),
    indexing='ij',
)
coords_all = torch.stack([ys, xs], dim=-1).view(cfg.n_pix, 2)
pix_all = img.permute(1, 2, 0).reshape(-1, 3)

tokenizer = Tokenizer(d_model=cfg.d_model, rgb_channels=3,
                       fourier_dim=cfg.fourier_dim, fourier_scale=cfg.fourier_scale)
all_tokens = tokenizer(pix_all.unsqueeze(0), coords_all.unsqueeze(0))
print(f"all_tokens shape: {tuple(all_tokens.shape)}  (B=1, N=1024, D={cfg.d_model})")

# Show token heatmap
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.imshow(all_tokens[0, :, :64].detach().numpy(), aspect='auto', cmap='RdBu_r')
ax.set_title(f"Token tensor (first 64 of {cfg.d_model} dims, all {cfg.n_pix} pixels)")
ax.set_xlabel("token dim"); ax.set_ylabel("pixel (row-major)")
plt.tight_layout(); plt.show()


## 2. The 4 serialization orders

The encoder picks one of these four 1-D orderings at each layer, then
runs BigBird block-sparse attention along that linear sequence. Across
layers, spatial neighbors that aren't close in one ordering become close
in another.


In [ ]:
grid = precompute_grid_orderings(cfg.image_size)
# `grid[name][i]` = rank of pixel `i` along that curve.
# To draw the curve, we need the *inverse*: pixel at each rank.
rank_to_pixel = {name: torch.argsort(r) for name, r in grid.items()}

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, name in zip(axes, ('z', 'z_rev', 'hilbert', 'hilbert_rev')):
    order = rank_to_pixel[name].numpy()
    # Coordinate of each pixel (in image pixel coords [0, 32))
    iy, ix = order // cfg.image_size, order % cfg.image_size
    # Color = rank along the curve
    rank = np.arange(cfg.n_pix)
    sc = ax.scatter(ix, iy, c=rank, cmap='turbo', s=18, edgecolors='none')
    # Draw the curve as a connected line, every 4th step for clarity
    ax.plot(ix[::4], iy[::4], color='black', lw=0.4, alpha=0.5)
    ax.invert_yaxis()
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(name, fontsize=12)
    ax.set_aspect('equal')
fig.suptitle("Four space-filling-curve orderings on the 32×32 grid\n"
             "(color = position along the curve)", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()


## 3. Per-sample subset orderings

For a sparse subset (e.g. the 100 context pixels), we just *look up* each
pixel's grid-rank and sort. O(K log K), no need to re-walk the curve.

Below: the same 100 context pixels, plotted along each of the 4 orderings.


In [ ]:
# Sample a context chunk (100 contiguous pool pixels)
rng = np.random.RandomState(7)
pool_idx = rng.permutation(cfg.n_pix)[:cfg.k_pool]
pool_xy  = coords_all[pool_idx]
a = rng.randint(cfg.k_pool)
d2 = (pool_xy - pool_xy[a]).pow(2).sum(-1).numpy()
ctx_local = np.argsort(d2, kind='stable')[:cfg.k_ctx]
ctx_idx = torch.from_numpy(pool_idx[ctx_local].astype(np.int64))
ctx_xy  = coords_all[ctx_idx]

subset_ords = precompute_subset_orderings(ctx_idx, grid)
print(f"ctx_idx shape: {tuple(ctx_idx.shape)}")
print(f"subset orderings:", list(subset_ords.keys()))

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, name in zip(axes, subset_ords.keys()):
    p = subset_ords[name]['perm'].numpy()
    rank = np.arange(cfg.k_ctx)
    # Show the context pixels in image coords, colored by their position in the order
    iy = (ctx_idx[p].numpy()) // cfg.image_size
    ix = (ctx_idx[p].numpy()) %  cfg.image_size
    sc = ax.scatter(ix, iy, c=rank, cmap='turbo', s=40, edgecolors='black', linewidths=0.3)
    ax.plot(ix, iy, color='black', lw=0.5, alpha=0.4)
    ax.set_xlim(-0.5, cfg.image_size - 0.5); ax.set_ylim(cfg.image_size - 0.5, -0.5)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{name}  ({cfg.k_ctx} ctx pixels)", fontsize=12)
    ax.set_aspect('equal')
fig.suptitle("100 context pixels ordered by each curve "
             "(color = position 0…99 in the curve order)",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()


## 4. BigBird block-sparse attention pattern

For sequence length `N`, blocks of `block_size`, each query block attends to:
- `(2W+1)` window blocks (clamped at boundaries)
- `n_global` global blocks (first and last)
- `n_random` random blocks (resampled each forward)

The total "attendees" per query block is `(2W+1) + G + R`. For `N=128`,
`block_size=32`, that's 4 blocks; with `W=1`, `G=2`, `R=2` → 3+2+2 = 7
attended blocks ⇒ but with only 4 blocks total this is over-coverage.
For longer sequences (e.g. `N=1024`, 32 blocks), `7/32 ≈ 22%` sparsity.


In [ ]:
def attention_mask_for_pattern(num_blocks, block_size, window, n_global, n_random,
                                  seed=0):
    # Build a dense (N,N) bool mask showing which keys each query attends to,
    # for visualization purposes only.
    torch.manual_seed(seed)
    g = torch.Generator(); g.manual_seed(seed)
    N = num_blocks * block_size
    mask = torch.zeros(N, N, dtype=torch.bool)
    # window
    for q in range(num_blocks):
        for w in range(max(0, q-window), min(num_blocks, q+window+1)):
            mask[q*block_size:(q+1)*block_size, w*block_size:(w+1)*block_size] = True
    # globals (assume first & last block)
    if n_global >= 1:
        mask[:, :block_size] = True
    if n_global >= 2:
        mask[:, -block_size:] = True
    # random (per query block)
    for q in range(num_blocks):
        rb = torch.randint(0, num_blocks, (n_random,), generator=g)
        for r in rb.tolist():
            mask[q*block_size:(q+1)*block_size, r*block_size:(r+1)*block_size] = True
    return mask

# 128-token toy
m128 = attention_mask_for_pattern(num_blocks=4, block_size=32,
                                   window=cfg.window, n_global=cfg.n_global,
                                   n_random=cfg.n_random, seed=0)
# 1024-token toy
m1024 = attention_mask_for_pattern(num_blocks=32, block_size=32,
                                    window=cfg.window, n_global=cfg.n_global,
                                    n_random=cfg.n_random, seed=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(m128.numpy(), cmap='Greys', aspect='auto', interpolation='nearest')
axes[0].set_title(f"BigBird mask  N=128  (4 blocks of 32)\n"
                   f"density = {m128.float().mean()*100:.1f}%")
axes[0].set_xlabel("key position"); axes[0].set_ylabel("query position")

axes[1].imshow(m1024.numpy(), cmap='Greys', aspect='auto', interpolation='nearest')
axes[1].set_title(f"BigBird mask  N=1024  (32 blocks of 32)\n"
                   f"density = {m1024.float().mean()*100:.1f}%")
axes[1].set_xlabel("key position"); axes[1].set_ylabel("query position")

plt.tight_layout(); plt.show()


## 5. Equivalence sanity: BigBird `full` mode == dense MHA

When `equivalent_to_dense=True`, BigBird bypasses the block-sparse path and
runs dense scaled-dot-product attention. With weights tied, the two layers
must produce identical outputs.


In [ ]:
# Tie weights between MHA and BigBird, then compare outputs
torch.manual_seed(0)
B, N, D = 2, 128, 64
H, Dh = 4, 16
x = torch.randn(B, N, D)
dense  = MultiHeadAttention(D, n_heads=H, dim_head=Dh)
sparse = BigBirdSparseAttention(D, n_heads=H, dim_head=Dh,
                                 block_size=32, window=1, n_random=2, n_global=2,
                                 equivalent_to_dense=True)
with torch.no_grad():
    sparse.to_qkv.weight.copy_(dense.to_qkv.weight)
    sparse.to_out.weight.copy_(dense.to_out.weight); sparse.to_out.bias.copy_(dense.to_out.bias)

y_d = dense(x); y_s = sparse(x)
max_abs = (y_d - y_s).abs().max().item()
print(f"BigBird(full) vs dense MHA: max|Δ| = {max_abs:.2e}  →  {'MATCH ✓' if max_abs < 1e-5 else 'MISMATCH ✗'}")

# And in sparse mode, outputs differ (we lost coverage)
sparse_sp = BigBirdSparseAttention(D, n_heads=H, dim_head=Dh,
                                    block_size=32, window=1, n_random=2, n_global=2,
                                    equivalent_to_dense=False)
with torch.no_grad():
    sparse_sp.to_qkv.weight.copy_(dense.to_qkv.weight)
    sparse_sp.to_out.weight.copy_(dense.to_out.weight); sparse_sp.to_out.bias.copy_(dense.to_out.bias)
y_sp = sparse_sp(x)
diff_sp = (y_d - y_sp).abs().max().item()
print(f"BigBird(sparse) vs dense MHA: max|Δ| = {diff_sp:.3f}  (should be > 0)")


## 6. Per-layer random shuffling in the encoder

Each `EncoderBlock` picks an ordering uniformly at random when called.
We visualize: a 6-layer encoder on the same 100-pixel context, showing
which ordering each layer chose.


### Why re-inject positional embedding at every layer?

The per-layer random reshuffling only motivates the model to attend to the
*true* positional embedding (`γ(coord)`) if that embedding is **available at
every layer**. Otherwise:

1. The tokenizer's positional signal gets mixed into content by layer 2 or 3.
2. BigBird attention is sequence-position-based (window / global / random in
   sequence order). Sequence position changes each layer because we reshuffle.
3. So a model could in principle learn to encode the *current layer's
   sequence position* into its content — but that representation is wrong
   for the next layer, which uses a different ordering. The training signal
   is noisy.

By re-adding `pos_emb = pos_proj(γ(coord))` as a residual at the start of
every encoder block (`reinject_pos=True`), every block always has access to
the *unchanging* true spatial position, regardless of how earlier layers
shuffled things. This is the kind of conditioning used by perceiver-style
cross-attention and (in spirit) by RoPE, simplified to an additive residual.

Set `cfg.reinject_pos = False` to ablate.


In [ ]:
enc = PBBEncoder(
    d_model=cfg.d_model, n_layers=cfg.n_layers_enc,
    n_heads=cfg.n_heads, dim_head=cfg.dim_head,
    block_size=cfg.block_size, window=cfg.window,
    n_random=cfg.n_random, n_global=cfg.n_global,
    ffn_mult=cfg.ffn_mult, fourier_dim=cfg.fourier_dim,
    fourier_scale=cfg.fourier_scale,
    serial_orders=cfg.serial_orders,
    reinject_pos=cfg.reinject_pos,
)
print(f"PBBEncoder params: {short_params(enc)}")

# Monkey-patch each block's forward to log the chosen order
chosen_orders = []
orig_forward = EncoderBlock.forward
def logged_forward(self, x, perm, inverse_perm, key_padding_mask=None):
    # Identify which ordering this perm corresponds to. We sniff by comparing
    # the first ~5 indices against each known perm.
    return orig_forward(self, x, perm, inverse_perm, key_padding_mask=key_padding_mask)
EncoderBlock.forward = logged_forward

# Quick forward — recover the order from the orderings dict
ctx_p_b = pix_all[ctx_idx].unsqueeze(0)
ctx_c_b = ctx_xy.unsqueeze(0)

# Build per-sample orderings dict for the encoder
ord_dict = {name: {"perm": subset_ords[name]["perm"].unsqueeze(0),
                    "inverse": subset_ords[name]["inverse"].unsqueeze(0)}
             for name in cfg.serial_orders}

# Instead of monkey-patching, manually run the encoder loop while recording order names
torch.manual_seed(1)
x = enc.tokenizer(ctx_p_b, ctx_c_b)
x, pm, K_orig = enc._pad_to_block_multiple(x, enc.block_size, None)
B, Kp, D = x.shape
extended = {}
for name, d in ord_dict.items():
    p, inv = d["perm"], d["inverse"]
    if p.shape[1] != Kp:
        tail = torch.arange(p.shape[1], Kp).unsqueeze(0).expand(B, -1)
        p = torch.cat([p, tail], dim=1); inv = torch.cat([inv, tail], dim=1)
    extended[name] = (p, inv)

order_names = list(cfg.serial_orders)
chosen = []
for blk in enc.blocks:
    nm = order_names[torch.randint(0, len(order_names), (1,)).item()]
    chosen.append(nm)
    p, inv = extended[nm]
    x = blk(x, p, inv, key_padding_mask=pm)
x = enc.norm(x)[:, :K_orig]
print(f"\nencoder output shape: {tuple(x.shape)}")
print(f"orderings used per layer: {chosen}")


## 7. End-to-end forward — encoder + predictor + target lookup

Demonstration on one image, mirroring the training step.


In [ ]:
# Build a full v8-style multi-block sample
rng = np.random.RandomState(11)
pool = rng.permutation(cfg.n_pix)[:cfg.k_pool]
pool_xy = coords_all[pool]
forbidden = np.zeros(cfg.k_pool, dtype=bool)
tgt_blocks_local = []
for _ in range(cfg.n_pred):
    a = rng.randint(cfg.k_pool)
    d2 = (pool_xy - pool_xy[a]).pow(2).sum(-1).numpy()
    blk = np.argsort(d2, kind='stable')[:cfg.k_tgt]
    tgt_blocks_local.append(blk); forbidden[blk] = True
tgt_local = np.concatenate(tgt_blocks_local).astype(np.int64)
allowed = np.where(~forbidden)[0]
a_ctx = allowed[rng.randint(len(allowed))]
d2 = (pool_xy[allowed] - pool_xy[a_ctx]).pow(2).sum(-1).numpy()
ctx_local = allowed[np.argsort(d2, kind='stable')[:cfg.k_ctx]]

ctx_idx_g  = torch.from_numpy(pool[ctx_local].astype(np.int64))
pool_idx_g = torch.from_numpy(pool.astype(np.int64))
tgt_pool_pos = torch.from_numpy(tgt_local.astype(np.int64))
tgt_idx_g  = torch.from_numpy(pool[tgt_local].astype(np.int64))

ctx_p = pix_all[ctx_idx_g].unsqueeze(0)
ctx_c = coords_all[ctx_idx_g].unsqueeze(0)
pool_p = pix_all[pool_idx_g].unsqueeze(0)
pool_c = coords_all[pool_idx_g].unsqueeze(0)
tgt_c  = coords_all[tgt_idx_g].unsqueeze(0)

ctx_ords  = {n: {"perm": d[0].unsqueeze(0), "inverse": d[1].unsqueeze(0)}
              for n, d in zip(grid.keys(), [
                  (subset_perm(ctx_idx_g,  grid[k]), invert_perm(subset_perm(ctx_idx_g,  grid[k])))
                  for k in grid.keys()])}
pool_ords = {n: {"perm": d[0].unsqueeze(0), "inverse": d[1].unsqueeze(0)}
              for n, d in zip(grid.keys(), [
                  (subset_perm(pool_idx_g, grid[k]), invert_perm(subset_perm(pool_idx_g, grid[k])))
                  for k in grid.keys()])}

target_encoder = copy.deepcopy(enc)
for q in target_encoder.parameters(): q.requires_grad_(False)
predictor = PBBPredictor(
    d_model=cfg.d_model, d_pred=cfg.d_pred, n_layers=cfg.n_layers_pred,
    n_heads=cfg.n_heads_pred, dim_head=cfg.dim_head_pred,
    fourier_dim=cfg.fourier_dim, fourier_scale=cfg.fourier_scale, ffn_mult=cfg.ffn_mult,
)

with torch.no_grad():
    g_tgt = target_encoder(pool_p, pool_c, pool_ords)
    h_tgt_raw = gather_target_features(g_tgt, tgt_pool_pos.unsqueeze(0))
    h_tgt = F.layer_norm(h_tgt_raw, (h_tgt_raw.size(-1),))

g_ctx = enc(ctx_p, ctx_c, ctx_ords)
h_pred = predictor(g_ctx, tgt_c)
loss = jepa_loss(h_pred, h_tgt)

print("end-to-end shapes:")
print(f"  g_ctx     : {tuple(g_ctx.shape)}    (context per-token features)")
print(f"  g_tgt     : {tuple(g_tgt.shape)}    (target per-token features over pool)")
print(f"  h_tgt_raw : {tuple(h_tgt_raw.shape)}     (gathered at target positions)")
print(f"  h_pred    : {tuple(h_pred.shape)}     (predictor output)")
print(f"\nuntrained loss = {loss.item():.4f}")
print(f"cos(h_pred, h_tgt) = {F.cosine_similarity(h_pred, h_tgt, dim=-1).mean().item():.3f}")


## 8. Data viz: 5-panel showing context + 4 disjoint target blocks

Same v8 visualization style as `JEPA_architecture.ipynb`, but now the
context block is K_CTX=100 (smaller) and each target block is 50 px (4 blocks).


In [ ]:
TGT_COLORS = ['#fbbf24', '#34d399', '#60a5fa', '#f472b6']
CTX_COLOR = '#ef4444'

def img_canvas(idx=None):
    im = ((img.permute(1,2,0)*0.5)+0.5).clamp(0,1).numpy()
    if idx is None: return im
    canvas = np.ones_like(im) * 0.85
    canvas.reshape(-1, 3)[idx] = im.reshape(-1, 3)[idx]
    return canvas

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(img_canvas(), extent=(-1,1,1,-1))
axes[0].set_title(f"(a) original\n{CIFAR_CLASSES[label]}", fontsize=10)

axes[1].imshow(img_canvas(pool), extent=(-1,1,1,-1))
axes[1].set_title(f"(b) 40% pool ({cfg.k_pool})", fontsize=10)

axes[2].imshow(img_canvas(), extent=(-1,1,1,-1), alpha=0.32)
for k, blk in enumerate(tgt_blocks_local):
    xy = coords_all[pool[blk]]
    axes[2].scatter(xy[:,1], xy[:,0], s=22, c=TGT_COLORS[k],
                    edgecolors='black', linewidths=0.3, label=f'block {k+1}')
axes[2].set_title(f"(c) {cfg.n_pred} target blocks first", fontsize=10)
axes[2].legend(loc='lower right', fontsize=6.5, ncol=2)

axes[3].imshow(img_canvas(), extent=(-1,1,1,-1), alpha=0.32)
ctx_xy_full = coords_all[pool[ctx_local]]
axes[3].scatter(ctx_xy_full[:,1], ctx_xy_full[:,0], s=22, c=CTX_COLOR,
                edgecolors='black', linewidths=0.3)
axes[3].set_title(f"(d) context block (disjoint)\nK_CTX={cfg.k_ctx}", fontsize=10)

axes[4].imshow(img_canvas(), extent=(-1,1,1,-1), alpha=0.30)
axes[4].scatter(ctx_xy_full[:,1], ctx_xy_full[:,0], s=14, c=CTX_COLOR, alpha=0.85,
                edgecolors='black', linewidths=0.2, label=f'ctx ({cfg.k_ctx})')
for k, blk in enumerate(tgt_blocks_local):
    xy = coords_all[pool[blk]]
    axes[4].scatter(xy[:,1], xy[:,0], s=22, c=TGT_COLORS[k],
                    edgecolors='black', linewidths=0.3, label=f'tgt {k+1}')
axes[4].set_title(f"(e) overlay — disjoint ✓", fontsize=10)
axes[4].legend(loc='lower right', fontsize=6, ncol=2)

for ax in axes:
    ax.set_xlim(-1.05, 1.05); ax.set_ylim(1.05, -1.05)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle("v8-style multi-block masking (carried over from JEPA_CIFAR10.ipynb)",
             fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()


## 9. Summary diagram (text)

```
─── Tokenization ──────────────────────────────────────────────
   per pixel n:  token_n = W_signal·rgb_n + W_pos·γ(coord_n)
   ⇒ (B, K, D)

─── Encoder (L layers) ────────────────────────────────────────
   for each layer:
     1) sample ordering from {z, z_rev, hilbert, hilbert_rev}
     2) x_perm = gather(x, perm)            # (B, K, D)
     3) y_perm = BigBirdSparseAttn(x_perm)  # window + global + random blocks
     4) y_perm = y_perm + FFN(LN(y_perm))
     5) x      = gather(y_perm, inverse)    # back to canonical order
   ⇒ g_ctx (B, K_ctx, D)
   ⇒ g_tgt (B, K_pool, D)  via the EMA copy of the encoder

─── Predictor (i-JEPA style) ──────────────────────────────────
   ctx_tok = proj_g(g_ctx)
   tgt_tok = proj_pos(γ(target_coords)) + mask_token
   x = concat([ctx_tok, tgt_tok])
   x = 4 × DenseTransformerBlock(x)
   h_pred = proj_out(LayerNorm(x[K_ctx:]))   # (B, N_tgt, D)

─── Target features (no learnable params on target side) ─────
   h_tgt = LayerNorm(center( gather(g_tgt, tgt_pool_pos) ))

─── Loss ──────────────────────────────────────────────────────
   L = smooth_L1(h_pred, h_tgt)
   target_encoder ← EMA(context_encoder), m: 0.999 → 1.0
```

The key novelty: per-layer random serialization means each token effectively
gets to attend to its full neighborhood graph **over the depth of the
network**, even though any single attention step is sparse along one
specific 1-D ordering.
